# P5 Mode Distinction Audit

This notebook implements **Phase P5** from the frozen pipeline.

## Goals

- compute mode coverage for the count and capacity specifications
- compute mode-level concentration diagnostics
- compute effect-size-style mode distinction metrics
- compute community disruption under leave-one-mode-out exclusion
- classify modes into `discriminative`, `background`, or `head-dominated`
- compare count and capacity mode-audit structures

## Required outputs

- `artifacts/tables/p5_mode_distinction_count.csv`
- `artifacts/tables/p5_mode_distinction_capacity.csv`
- `artifacts/tables/p5_mode_distinction_classes.csv`
- `artifacts/reports/p5_mode_distinction_report.md`

Additional support output generated here:

- `artifacts/runtime/p5_mode_distinction_manifest.json`

This notebook uses the frozen `P2` row-normalized matrices and `P3` community assignments as its inputs.


In [ ]:
from pathlib import Path
import json
import math

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
if not (ROOT / 'artifacts' / 'tables' / 'p2_country_mode_count_row_normalized.csv').exists():
    ROOT = ROOT.parent

TABLES = ROOT / 'artifacts' / 'tables'
REPORTS = ROOT / 'artifacts' / 'reports'
RUNTIME = ROOT / 'artifacts' / 'runtime'

for path in [TABLES, REPORTS, RUNTIME]:
    path.mkdir(parents=True, exist_ok=True)

PIPELINE_VERSION = 'A1_GSTP2026Feb_contract_v1'
RUN_ID = 'A1_GSTP2026Feb_contract_v1__20260320__P5__nogit'
SEED = 42
RESOLUTION = 1.0
TOP_K = 6
SIMILARITY = 'cosine'
EXPECTED_COUNTRIES = 94
EXPECTED_MODES = 16
HEAD_DOMINATED_TOP10_THRESHOLD = 0.80
DISCRIMINATIVE_SALIENCE_THRESHOLD = 0.70
ACTIVE_MODE_COVERAGE_THRESHOLD_PCT = 5.0
SPARSE_ZERO_THRESHOLD_PCT = 90.0

manifest = {
    'notebook': 'P5_Mode_Distinction_Audit.ipynb',
    'pipeline_version': PIPELINE_VERSION,
    'run_id': RUN_ID,
    'inputs': {
        'count_row_normalized': str(TABLES / 'p2_country_mode_count_row_normalized.csv'),
        'capacity_row_normalized': str(TABLES / 'p2_country_mode_capacity_row_normalized.csv'),
        'count_membership': str(TABLES / 'p3_count_community_membership.csv'),
        'capacity_membership': str(TABLES / 'p3_capacity_community_membership.csv'),
    },
    'classification_rules': {
        'head_dominated_top10_country_share_threshold': HEAD_DOMINATED_TOP10_THRESHOLD,
        'discriminative_salience_threshold': DISCRIMINATIVE_SALIENCE_THRESHOLD,
        'active_mode_coverage_threshold_pct': ACTIVE_MODE_COVERAGE_THRESHOLD_PCT,
        'sparse_zero_threshold_pct': SPARSE_ZERO_THRESHOLD_PCT,
    },
}
(RUNTIME / 'p5_mode_distinction_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')

display(Markdown(f"**Run ID:** `{RUN_ID}`"))
display(Markdown('**Inputs:** `P2` row-normalized matrices + `P3` community memberships'))


## Load Inputs and Verify Community Alignment

The mode audit should be run on the same row-normalized country-mode matrices and the same community assignments used in the frozen `P3` path.


In [ ]:
count_row = pd.read_csv(TABLES / 'p2_country_mode_count_row_normalized.csv').set_index('Country/Area')
capacity_row = pd.read_csv(TABLES / 'p2_country_mode_capacity_row_normalized.csv').set_index('Country/Area')
count_membership = pd.read_csv(TABLES / 'p3_count_community_membership.csv').set_index('Country/Area')
capacity_membership = pd.read_csv(TABLES / 'p3_capacity_community_membership.csv').set_index('Country/Area')

for name, matrix in {'count_row': count_row, 'capacity_row': capacity_row}.items():
    if matrix.shape != (EXPECTED_COUNTRIES, EXPECTED_MODES):
        raise ValueError(f'{name} shape drift: expected {(EXPECTED_COUNTRIES, EXPECTED_MODES)}, got {matrix.shape}')
    if matrix.isna().any().any():
        raise ValueError(f'{name} contains NaN values.')
    if (matrix < 0).any().any():
        raise ValueError(f'{name} contains negative values.')
    if not np.isclose(matrix.sum(axis=1).to_numpy(dtype=float), 1.0).all():
        raise ValueError(f'{name} is not properly row-normalized.')

for name, membership, matrix in [
    ('count', count_membership, count_row),
    ('capacity', capacity_membership, capacity_row),
]:
    if membership['community'].isna().any():
        raise ValueError(f'{name} membership contains missing community labels.')
    if sorted(membership['community'].unique().tolist()) != list(range(1, membership['community'].nunique() + 1)):
        raise ValueError(f'{name} community labels are not consecutively relabeled.')
    if set(membership.index) != set(matrix.index):
        raise ValueError(f'{name} membership does not cover the same country set as the matrix.')

input_check = pd.DataFrame({
    'object': ['count_row', 'capacity_row', 'count_membership', 'capacity_membership'],
    'rows': [len(count_row), len(capacity_row), len(count_membership), len(capacity_membership)],
    'cols_or_communities': [count_row.shape[1], capacity_row.shape[1], count_membership['community'].nunique(), capacity_membership['community'].nunique()],
})
display(input_check)


## Helper Functions

This section computes the approved `P5` audit families:

- coverage
- concentration
- effect-size-style distinction (`eta_squared`)
- community disruption under leave-one-mode-out exclusion
- edge-contribution share
- combined salience score


In [ ]:
def split_mode(mode):
    return mode.split(' | ', 1)


def row_normalize(matrix):
    row_sums = matrix.sum(axis=1).replace(0, np.nan)
    return matrix.div(row_sums, axis=0).fillna(0)


def build_projection_graph(matrix, similarity='cosine', top_k=6):
    names = list(matrix.index)
    values = matrix.to_numpy(dtype=float)
    graph = nx.Graph()
    graph.add_nodes_from(names)

    if len(names) < 2:
        return graph

    if similarity == 'cosine':
        sim = cosine_similarity(values)
    elif similarity == 'dot':
        sim = values @ values.T
    else:
        raise ValueError(f'Unknown similarity: {similarity}')

    np.fill_diagonal(sim, 0)
    k = min(top_k, len(names) - 1)

    for i, source in enumerate(names):
        neighbor_idx = np.argsort(sim[i])[-k:]
        for j in neighbor_idx:
            if i == j:
                continue
            weight = float(sim[i, j])
            if weight <= 0:
                continue
            target = names[j]
            if graph.has_edge(source, target):
                graph[source][target]['weight'] = max(graph[source][target]['weight'], weight)
            else:
                graph.add_edge(source, target, weight=weight)

    return graph


def detect_communities(graph, seed=42, resolution=1.0):
    communities = list(nx.community.louvain_communities(graph, weight='weight', seed=seed, resolution=resolution))
    communities = sorted(communities, key=len, reverse=True)
    assignment = pd.Series(
        {node: cid for cid, members in enumerate(communities, start=1) for node in members},
        name='community',
    ).sort_index()
    modularity = float(nx.community.modularity(graph, communities, weight='weight'))
    return assignment, modularity


def eta_squared(values, groups):
    x = np.asarray(values, dtype=float)
    g = np.asarray(groups)
    mask = np.isfinite(x) & pd.notna(g)
    x = x[mask]
    g = g[mask]
    if len(x) <= 1:
        return np.nan
    grand_mean = x.mean()
    ss_total = ((x - grand_mean) ** 2).sum()
    if ss_total <= 0:
        return 0.0
    ss_between = 0.0
    for label in pd.unique(g):
        xg = x[g == label]
        if len(xg) == 0:
            continue
        ss_between += len(xg) * (xg.mean() - grand_mean) ** 2
    return float(ss_between / ss_total)


def classify_mode(row):
    if row['top10_country_share'] >= HEAD_DOMINATED_TOP10_THRESHOLD:
        return 'head-dominated'
    if row['coverage_pct'] >= ACTIVE_MODE_COVERAGE_THRESHOLD_PCT and row['salience_score'] >= DISCRIMINATIVE_SALIENCE_THRESHOLD:
        return 'discriminative'
    return 'background'


def audit_mode_distinction(weight_kind, matrix_row, assignment):
    assignment = assignment.reindex(matrix_row.index)
    graph = build_projection_graph(matrix_row, similarity=SIMILARITY, top_k=TOP_K)
    _, baseline_modularity = detect_communities(graph, seed=SEED, resolution=RESOLUTION)
    edge_total = float(sum(d['weight'] for _, _, d in graph.edges(data=True)))
    mode_index = {mode: idx for idx, mode in enumerate(matrix_row.columns)}
    rows = []

    for mode in matrix_row.columns:
        values = matrix_row[mode].astype(float)
        ranked = values.sort_values(ascending=False)
        coverage_pct = 100 * float((values > 0).mean())
        zero_country_pct = 100 - coverage_pct
        by_comm = values.groupby(assignment).mean().sort_values(ascending=False)
        dominant_community = int(by_comm.index[0])
        dominant_community_mean_share = float(by_comm.iloc[0])
        global_mean_share = float(values.mean())
        community_lift = dominant_community_mean_share - global_mean_share

        reduced_matrix = row_normalize(matrix_row.drop(columns=[mode]))
        reduced_graph = build_projection_graph(reduced_matrix, similarity=SIMILARITY, top_k=TOP_K)
        reduced_assignment, reduced_modularity = detect_communities(reduced_graph, seed=SEED, resolution=RESOLUTION)
        common = assignment.index.intersection(reduced_assignment.index)
        ari_without_mode = adjusted_rand_score(assignment.loc[common], reduced_assignment.loc[common]) if len(common) > 1 else np.nan
        community_disruption = float(1 - ari_without_mode) if pd.notna(ari_without_mode) else np.nan
        modularity_delta_without_mode = float(baseline_modularity - reduced_modularity)

        edge_contribution = 0.0
        idx = mode_index[mode]
        for u, v, d in graph.edges(data=True):
            products = matrix_row.loc[u].to_numpy(dtype=float) * matrix_row.loc[v].to_numpy(dtype=float)
            dot = float(products.sum())
            if dot <= 0:
                continue
            edge_contribution += float(d['weight'] * (products[idx] / dot))
        edge_contribution_share = float(edge_contribution / edge_total) if edge_total > 0 else np.nan

        rows.append({
            'weight_kind': weight_kind,
            'mode': mode,
            'status': split_mode(mode)[0],
            'capacity_tier': split_mode(mode)[1],
            'coverage_pct': coverage_pct,
            'zero_country_pct': zero_country_pct,
            'active_over_5pct_coverage': bool(coverage_pct >= ACTIVE_MODE_COVERAGE_THRESHOLD_PCT),
            'sparse_over_90pct_zero': bool(zero_country_pct > SPARSE_ZERO_THRESHOLD_PCT),
            'top3_country_share': float(ranked.head(3).sum() / ranked.sum()) if ranked.sum() > 0 else np.nan,
            'top10_country_share': float(ranked.head(10).sum() / ranked.sum()) if ranked.sum() > 0 else np.nan,
            'eta2': eta_squared(values, assignment),
            'dominant_community': dominant_community,
            'dominant_community_mean_share': dominant_community_mean_share,
            'global_mean_share': global_mean_share,
            'community_lift': community_lift,
            'ari_without_mode': float(ari_without_mode) if pd.notna(ari_without_mode) else np.nan,
            'community_disruption': community_disruption,
            'modularity_delta_without_mode': modularity_delta_without_mode,
            'edge_contribution_share': edge_contribution_share,
        })

    out = pd.DataFrame(rows)
    for metric in ['eta2', 'community_disruption', 'edge_contribution_share', 'community_lift']:
        out[f'{metric}_pct_rank'] = out[metric].rank(pct=True, ascending=True, method='average')
    out['salience_score'] = out[[
        'eta2_pct_rank',
        'community_disruption_pct_rank',
        'edge_contribution_share_pct_rank',
        'community_lift_pct_rank',
    ]].mean(axis=1)
    out['salience_rank'] = out['salience_score'].rank(ascending=False, method='dense')
    out['mode_class'] = out.apply(classify_mode, axis=1)
    return out.sort_values(['mode_class', 'salience_score', 'eta2'], ascending=[True, False, False]).reset_index(drop=True)


## Run the Mode Distinction Audit

The count and capacity paths are audited separately and then merged into a single comparison table.


In [ ]:
count_audit = audit_mode_distinction('count', count_row, count_membership['community'])
capacity_audit = audit_mode_distinction('capacity', capacity_row, capacity_membership['community'])

count_audit.to_csv(TABLES / 'p5_mode_distinction_count.csv', index=False)
capacity_audit.to_csv(TABLES / 'p5_mode_distinction_capacity.csv', index=False)

classes_compare = count_audit[[
    'mode', 'status', 'capacity_tier', 'mode_class', 'salience_rank', 'salience_score',
    'coverage_pct', 'top10_country_share', 'eta2', 'community_disruption', 'edge_contribution_share'
]].merge(
    capacity_audit[[
        'mode', 'mode_class', 'salience_rank', 'salience_score',
        'coverage_pct', 'top10_country_share', 'eta2', 'community_disruption', 'edge_contribution_share'
    ]],
    on='mode',
    suffixes=('_count', '_capacity'),
)
classes_compare['class_agreement'] = classes_compare['mode_class_count'] == classes_compare['mode_class_capacity']
classes_compare['discriminative_in_both'] = (classes_compare['mode_class_count'] == 'discriminative') & (classes_compare['mode_class_capacity'] == 'discriminative')
classes_compare['head_dominated_in_both'] = (classes_compare['mode_class_count'] == 'head-dominated') & (classes_compare['mode_class_capacity'] == 'head-dominated')
classes_compare['count_only_discriminative'] = (classes_compare['mode_class_count'] == 'discriminative') & (classes_compare['mode_class_capacity'] != 'discriminative')
classes_compare['capacity_only_discriminative'] = (classes_compare['mode_class_capacity'] == 'discriminative') & (classes_compare['mode_class_count'] != 'discriminative')
classes_compare['salience_rank_gap'] = (classes_compare['salience_rank_count'] - classes_compare['salience_rank_capacity']).abs()
classes_compare['shared_priority'] = classes_compare[['salience_score_count', 'salience_score_capacity']].mean(axis=1)
classes_compare = classes_compare.sort_values(['shared_priority', 'mode'], ascending=[False, True]).reset_index(drop=True)
classes_compare.to_csv(TABLES / 'p5_mode_distinction_classes.csv', index=False)

if len(count_audit) != EXPECTED_MODES or len(capacity_audit) != EXPECTED_MODES:
    raise ValueError('Mode audit does not cover all 16 modes for both paths.')
if int(count_audit['active_over_5pct_coverage'].sum()) != EXPECTED_MODES or int(capacity_audit['active_over_5pct_coverage'].sum()) != EXPECTED_MODES:
    raise ValueError('One or more modes no longer satisfy the active-mode threshold under the frozen baseline.')
if int(count_audit['sparse_over_90pct_zero'].sum()) != 0 or int(capacity_audit['sparse_over_90pct_zero'].sum()) != 0:
    raise ValueError('Unexpected >90% sparse mode detected under the frozen baseline.')

comparison_summary = pd.DataFrame([
    {
        'weight_kind': 'count',
        'discriminative_modes': int((count_audit['mode_class'] == 'discriminative').sum()),
        'background_modes': int((count_audit['mode_class'] == 'background').sum()),
        'head_dominated_modes': int((count_audit['mode_class'] == 'head-dominated').sum()),
        'top_salience_mode': count_audit.sort_values('salience_score', ascending=False)['mode'].iloc[0],
    },
    {
        'weight_kind': 'capacity',
        'discriminative_modes': int((capacity_audit['mode_class'] == 'discriminative').sum()),
        'background_modes': int((capacity_audit['mode_class'] == 'background').sum()),
        'head_dominated_modes': int((capacity_audit['mode_class'] == 'head-dominated').sum()),
        'top_salience_mode': capacity_audit.sort_values('salience_score', ascending=False)['mode'].iloc[0],
    },
])

display(Markdown('**Mode-audit class counts**'))
display(comparison_summary)
display(Markdown('**Shared comparison table**'))
display(classes_compare[['mode', 'mode_class_count', 'mode_class_capacity', 'salience_rank_count', 'salience_rank_capacity', 'shared_priority']])


## Write the P5 Report

The report documents the class rules and summarizes which modes appear to carry most cross-national differentiation signal.


In [ ]:
salience_spearman = float(classes_compare['salience_rank_count'].corr(classes_compare['salience_rank_capacity'], method='spearman'))
shared_discriminative_modes = classes_compare.loc[classes_compare['discriminative_in_both'], 'mode'].tolist()
shared_head_dominated_modes = classes_compare.loc[classes_compare['head_dominated_in_both'], 'mode'].tolist()
count_only_discriminative_modes = classes_compare.loc[classes_compare['count_only_discriminative'], 'mode'].tolist()
capacity_only_discriminative_modes = classes_compare.loc[classes_compare['capacity_only_discriminative'], 'mode'].tolist()

report_lines = [
    '# P5 Mode Distinction Report',
    '',
    '## Run metadata',
    f'- run_id: `{RUN_ID}`',
    f'- pipeline_version: `{PIPELINE_VERSION}`',
    '- inputs: `P2` row-normalized matrices and `P3` community assignments',
    '',
    '## Audit rules used',
    f'- head-dominated rule: `top10_country_share >= {HEAD_DOMINATED_TOP10_THRESHOLD:.2f}`',
    f'- discriminative rule: active coverage >= {ACTIVE_MODE_COVERAGE_THRESHOLD_PCT:.1f}% and `salience_score >= {DISCRIMINATIVE_SALIENCE_THRESHOLD:.2f}`',
    '- background rule: modes that are neither head-dominated nor discriminative',
    '- effect-size-style metric: `eta_squared` between mode weight and community assignment',
    '- disruption metric: `1 - ARI` after leave-one-mode-out exclusion',
    '',
    '## Baseline QA check',
    f'- all 16 modes remain active in both count and capacity views under the frozen baseline',
    f'- no mode exceeds the >90% zero sparse threshold in either view',
    '',
    '## Class counts',
]
for row in comparison_summary.itertuples(index=False):
    report_lines.append(
        f'- {row.weight_kind}: discriminative={row.discriminative_modes}, background={row.background_modes}, head-dominated={row.head_dominated_modes}, top_salience_mode={row.top_salience_mode}'
    )

report_lines.extend([
    '',
    '## Count discriminative modes',
    '- ' + ', '.join(count_audit.loc[count_audit['mode_class'] == 'discriminative', 'mode'].tolist()),
    '',
    '## Capacity discriminative modes',
    '- ' + ', '.join(capacity_audit.loc[capacity_audit['mode_class'] == 'discriminative', 'mode'].tolist()),
    '',
    '## Shared structure',
    f'- salience-rank Spearman between count and capacity: {salience_spearman:.3f}',
    '- discriminative in both: ' + (', '.join(shared_discriminative_modes) if shared_discriminative_modes else 'none'),
    '- head-dominated in both: ' + (', '.join(shared_head_dominated_modes) if shared_head_dominated_modes else 'none'),
    '- count-only discriminative: ' + (', '.join(count_only_discriminative_modes) if count_only_discriminative_modes else 'none'),
    '- capacity-only discriminative: ' + (', '.join(capacity_only_discriminative_modes) if capacity_only_discriminative_modes else 'none'),
    '',
    '## Interpretation boundary',
    '- The audit does not imply that all 16 modes are equally important.',
    '- It supports the claim that a limited subset of modes carries most cross-national differentiation signal.',
    '- Head-dominated modes are flagged for interpretive caution rather than automatic exclusion.',
    '',
    '## Output inventory',
    '- `artifacts/tables/p5_mode_distinction_count.csv`',
    '- `artifacts/tables/p5_mode_distinction_capacity.csv`',
    '- `artifacts/tables/p5_mode_distinction_classes.csv`',
    '- `artifacts/reports/p5_mode_distinction_report.md`',
    '- `artifacts/runtime/p5_mode_distinction_manifest.json`',
])

(REPORTS / 'p5_mode_distinction_report.md').write_text('\n'.join(report_lines), encoding='utf-8')
display(Markdown((REPORTS / 'p5_mode_distinction_report.md').read_text(encoding='utf-8')))
